# 1.安装依赖

In [1]:
!python -m pip install -q -U --no-cache-dir "google-adk[extensions]" litellm openai requests python-dotenv pandas

print("✅ Dependencies installed.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
camelot-py 1.0.9 requires pypdf<4.0,>=3.17; python_version < "3.12", but you have pypdf 6.10.2 which is incompatible.
moviepy 2.2.1 requires pillow<12.0,>=9.2.0, but you have pillow 12.1.0 which is incompatible.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.
✅ Dependencies installed.


# 2.读取NVIDIA API Ket

In [2]:
import os
import getpass

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
    if not NVIDIA_API_KEY:
        raise ValueError("Colab Secret 'NVIDIA_API_KEY' is empty.")
    print("✅ Loaded NVIDIA_API_KEY from Colab Secrets.")
except Exception as e:
    print(f"⚠️ Could not read Colab Secret automatically: {e}")
    NVIDIA_API_KEY = getpass.getpass("Please paste your NVIDIA_API_KEY here: ").strip()
    print("✅ Loaded NVIDIA_API_KEY from manual input.")

# NVIDIA hosted OpenAI-compatible endpoint
NVIDIA_API_BASE = "https://integrate.api.nvidia.com/v1"

# OpenAI-compatible SDKs / LiteLLM often read OPENAI_API_KEY
os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
os.environ["OPENAI_API_KEY"] = NVIDIA_API_KEY

print("✅ Environment variables prepared.")
print("NVIDIA_API_BASE =", NVIDIA_API_BASE)

⚠️ Could not read Colab Secret automatically: No module named 'google.colab'
✅ Loaded NVIDIA_API_KEY from manual input.
✅ Environment variables prepared.
NVIDIA_API_BASE = https://integrate.api.nvidia.com/v1


# 3.列出当前 endpoint 可用的模型

In [3]:
import requests
import pandas as pd

headers = {
    "Authorization": f"Bearer {os.environ['NVIDIA_API_KEY']}"
}

resp = requests.get(
    f"{NVIDIA_API_BASE}/models",
    headers=headers,
    timeout=30
)
resp.raise_for_status()

models_payload = resp.json()
model_rows = models_payload.get("data", [])
model_ids = [row.get("id") for row in model_rows if row.get("id")]

print(f"✅ Found {len(model_ids)} models from NVIDIA endpoint.")
print("First 20 model ids:")
for mid in model_ids[:20]:
    print(" -", mid)

df_models = pd.DataFrame(model_rows)
df_models.head()

✅ Found 133 models from NVIDIA endpoint.
First 20 model ids:
 - 01-ai/yi-large
 - abacusai/dracarys-llama-3.1-70b-instruct
 - adept/fuyu-8b
 - ai21labs/jamba-1.5-large-instruct
 - aisingapore/sea-lion-7b-instruct
 - baai/bge-m3
 - bigcode/starcoder2-15b
 - bytedance/seed-oss-36b-instruct
 - databricks/dbrx-instruct
 - deepseek-ai/deepseek-coder-6.7b-instruct
 - deepseek-ai/deepseek-v3.1-terminus
 - deepseek-ai/deepseek-v3.2
 - google/codegemma-1.1-7b
 - google/codegemma-7b
 - google/deplot
 - google/gemma-2-2b-it
 - google/gemma-2b
 - google/gemma-3-12b-it
 - google/gemma-3-27b-it
 - google/gemma-3-4b-it


,id,object,created,owned_by
0,01-ai/yi-large,model,735790403,01-ai
1,abacusai/dracarys-llama-3.1-70b-instruct,model,735790403,abacusai
2,adept/fuyu-8b,model,735790403,adept
3,ai21labs/jamba-1.5-large-instruct,model,735790403,ai21labs
4,aisingapore/sea-lion-7b-instruct,model,735790403,aisingapore


# 4.自动选择一个模型

In [4]:
preferred_candidates = [
    "openai/gpt-oss-120b",
    "nvidia/llama-3.3-nemotron-super-49b-v1.5",
    "meta/llama-3.1-70b-instruct",
    "meta/llama-3.1-8b-instruct",
]

NVIDIA_MODEL_NAME = None

for candidate in preferred_candidates:
    if candidate in model_ids:
        NVIDIA_MODEL_NAME = candidate
        break

if NVIDIA_MODEL_NAME is None:
    if not model_ids:
        raise RuntimeError("No models returned by NVIDIA /v1/models.")
    NVIDIA_MODEL_NAME = model_ids[0]

print("✅ Selected NVIDIA model:", NVIDIA_MODEL_NAME)

✅ Selected NVIDIA model: openai/gpt-oss-120b


# 5.先直连测试 NVIDIA API

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url=NVIDIA_API_BASE,
    api_key=os.environ["NVIDIA_API_KEY"],
)

test_response = client.chat.completions.create(
    model=NVIDIA_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Reply with one sentence: confirm the API connection is working."},
    ],
    temperature=0.2,
    max_tokens=80,
)

print("✅ Direct NVIDIA API test succeeded.")
print(test_response.choices[0].message.content)

✅ Direct NVIDIA API test succeeded.
The API connection is working correctly.


# 6.导入 ADK + LiteLlm

In [6]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner

print("✅ ADK + LiteLLM imports succeeded.")

/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


✅ ADK + LiteLLM imports succeeded.


/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


# 7.定义天气工具
这版不用 Google Search，而是用一个自定义工具 get_weather(city) 来演示 tool calling。它先调用 Open-Meteo 的 Geocoding API 把城市名转经纬度，再调用 Forecast API 读当前天气。Open-Meteo 官方文档给出了 https://geocoding-api.open-meteo.com/v1/search，首页也说明无需 API key。

In [7]:
import requests
from typing import Dict, Any

WEATHER_CODE_MAP = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snow",
    73: "Moderate snow",
    75: "Heavy snow",
    80: "Rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    95: "Thunderstorm",
}

def get_weather(city: str) -> Dict[str, Any]:
    """
    Get current weather for a city using Open-Meteo.
    No extra API key required.
    """
    # 1) City name -> lat/lon
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json",
    }

    geo_resp = requests.get(geo_url, params=geo_params, timeout=30)
    geo_resp.raise_for_status()
    geo_data = geo_resp.json()

    results = geo_data.get("results") or []
    if not results:
        return {
            "status": "error",
            "message": f"Could not find city: {city}",
        }

    place = results[0]
    lat = place["latitude"]
    lon = place["longitude"]
    resolved_name = place.get("name", city)
    country = place.get("country", "")
    admin1 = place.get("admin1", "")

    # 2) Forecast API -> current weather
    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,apparent_temperature,wind_speed_10m,weather_code",
        "timezone": "auto",
    }

    weather_resp = requests.get(weather_url, params=weather_params, timeout=30)
    weather_resp.raise_for_status()
    weather_data = weather_resp.json()

    current = weather_data.get("current", {})
    weather_code = current.get("weather_code")

    return {
        "status": "success",
        "location": {
            "city": resolved_name,
            "admin1": admin1,
            "country": country,
            "latitude": lat,
            "longitude": lon,
        },
        "current_weather": {
            "temperature_2m": current.get("temperature_2m"),
            "apparent_temperature": current.get("apparent_temperature"),
            "wind_speed_10m": current.get("wind_speed_10m"),
            "weather_code": weather_code,
            "weather_text": WEATHER_CODE_MAP.get(weather_code, "Unknown"),
            "time": current.get("time"),
        },
    }

# 小测试
tool_test = get_weather("London")
print("✅ Weather tool test result:")
print(tool_test)

✅ Weather tool test result:
{'status': 'success', 'location': {'city': 'London', 'admin1': 'England', 'country': 'United Kingdom', 'latitude': 51.50853, 'longitude': -0.12574}, 'current_weather': {'temperature_2m': 11.5, 'apparent_temperature': 8.3, 'wind_speed_10m': 15.1, 'weather_code': 1, 'weather_text': 'Mainly clear', 'time': '2026-04-23T09:30'}}


# 8.定义 NVIDIA API 版 Agent

In [8]:
root_agent = LlmAgent(
    name="helpful_nvidia_agent",
    model=LiteLlm(
        model=f"openai/{NVIDIA_MODEL_NAME}",
        api_base=NVIDIA_API_BASE,
        api_key=os.environ["NVIDIA_API_KEY"],
    ),
    description="A helpful agent powered by an NVIDIA OpenAI-compatible endpoint.",
    instruction=(
        "You are a helpful assistant powered by NVIDIA API. "
        "Use the available tools whenever a question requires real-time or external data, "
        "such as current weather. "
        "When a tool returns structured data, explain it clearly in natural language."
    ),
    tools=[get_weather],
)

print("✅ NVIDIA ADK agent defined.")
print(root_agent)

✅ NVIDIA ADK agent defined.
name='helpful_nvidia_agent' description='A helpful agent powered by an NVIDIA OpenAI-compatible endpoint.' parent_agent=None sub_agents=[] before_agent_callback=None after_agent_callback=None model=LiteLlm(model='openai/openai/gpt-oss-120b', llm_client=<google.adk.models.lite_llm.LiteLLMClient object at 0x7f9604bdcf50>) instruction='You are a helpful assistant powered by NVIDIA API. Use the available tools whenever a question requires real-time or external data, such as current weather. When a tool returns structured data, explain it clearly in natural language.' global_instruction='' static_instruction=None tools=[<function get_weather at 0x7f960ae76de0>] generate_content_config=None disallow_transfer_to_parent=False disallow_transfer_to_peers=False include_contents='default' input_schema=None output_schema=None output_key=None planner=None code_executor=None before_model_callback=None after_model_callback=None on_model_error_callback=None before_tool_callb

# 9.创建 Runner

In [ ]:
runner = InMemoryRun ner(agent=root_agent)

print("✅ Runner created.")

✅ Runner created.


# 10.运行一个普通问题

In [10]:
response = await runner.run_debug(
    "What is an AI agent? Explain it for a beginner in 4 bullet points."
)

print("\n✅ First agent query finished.")
print("Returned object type:", type(response))


 ### Created new session: debug_session_id

User > What is an AI agent? Explain it for a beginner in 4 bullet points.
helpful_nvidia_agent > User asks: definition of AI agent, explained for beginner, 4 bullet points. No external data needed. Provide concise bullet list.**AI Agent – A Beginner‑Friendly Overview**

- **Software “personality” that can act** – An AI agent is a computer program designed to perceive its environment (through data, sensors, or user input) and then take actions or make decisions on its own, much like a virtual assistant or a robot.

- **Goal‑oriented behavior** – It’s built with a specific objective in mind (e.g., answering questions, recommending a movie, navigating a robot). The agent continuously evaluates what to do next to get closer to that goal.

- **Learning & adaptation** – Many AI agents use machine‑learning models that improve over time from experience, so they become better at tasks such as recognizing speech, translating languages, or playing game

# 11.运行一个会触发工具调用的问题

In [11]:
response = await runner.run_debug(
    "What's the weather in London right now? "
    "Please also tell me the apparent temperature and wind speed."
)

print("\n✅ Second agent query finished.")
print("Returned object type:", type(response))


 ### Continue session: debug_session_id

User > What's the weather in London right now? Please also tell me the apparent temperature and wind speed.
helpful_nvidia_agent > User asks current weather in London, need apparent temperature and wind speed. Use get_weather tool. Let's call with city "London".
helpful_nvidia_agent > Here’s the latest weather for **London, United Kingdom** (as of 09:30 UTC on 23 April 2026):

| Parameter | Value |
|-----------|-------|
| **Temperature (air)** | **11.5 °C** |
| **Apparent temperature** (how it feels) | **8.3 °C** |
| **Wind speed** (10 m above ground) | **15.1 km/h** |
| **Condition** | Mainly clear |

So, it’s a bit cool and feels slightly colder because of the wind, which is blowing at about 15 km/h.

✅ Second agent query finished.
Returned object type: <class 'list'>
